# 🏆 Day 16 — Capstone: AI Research Assistant

**LangGraph from Zero to Advanced · Day 16 of 16**

**Today's goal:** build one complete app that uses **everything** from the course.

```
START ► plan ► human_review ─Send×N─► research ► write ► review ─ pass ─► END
                                                   ▲          │
                                                   └─ revise ─┘ (max 2)
```

| Feature | Learned on |
|---|---|
| State, reducers, routing, loops | Days 1–3 |
| OpenAI from `.env`, messages | Day 4–5 |
| Retriever as a knowledge source (RAG) | Day 13 |
| Structured output (plan, review) | Day 10 |
| Human approval of the plan (`interrupt`) | Day 8 |
| Parallel research workers (`Send`) | Day 11 |
| Evaluator–optimizer loop | Day 10 |
| Checkpointer + `thread_id` | Day 7 |
| Runtime context (reader level) | Day 14 |
| Custom progress streaming | Day 9 |
| Retry policy | Day 15 |

Build it step by step: run each cell and read the comments.

In [ ]:
%pip install -q -U langgraph langchain-openai langchain-text-splitters python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Step 1 — The knowledge base

Our assistant researches from a small library of notes about building AI agents (in real life: your PDFs, wiki, or a web-search tool).

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

library = {
    "agents.md": "An AI agent is an LLM that decides which actions to take in a loop. It calls tools, reads the "
                 "results and decides again until the task is done. Agents are flexible but less predictable than "
                 "fixed workflows, so use them only when the steps cannot be known in advance.",
    "tools.md": "Tools let an LLM use calculators, APIs and databases. Each tool needs a clear name, typed "
                "arguments and a good description. Tool errors should be returned to the model as messages so it "
                "can recover instead of crashing the app.",
    "memory.md": "Short-term memory keeps the current conversation using a checkpointer and a thread id. "
                 "Long-term memory stores facts about a user across conversations in a store. Long histories "
                 "should be trimmed or summarized to control cost.",
    "reliability.md": "Reliable agents use step limits to stop endless loops, retry policies for flaky APIs, "
                      "structured output for decisions, and human approval before risky actions such as "
                      "payments or sending emails. Tracing tools like LangSmith help debug each step.",
    "evaluation.md": "Evaluate agents with test questions and expected behaviours. An evaluator model can grade "
                     "answers for correctness and tone. Track cost, latency and success rate over time.",
    "rag.md": "Retrieval-augmented generation finds relevant document chunks with embeddings and gives them to "
              "the model as context. Good RAG systems cite sources, say 'I don't know' when context is missing, "
              "and rewrite unclear queries.",
}
chunks = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30).split_documents(
    [Document(page_content=t, metadata={"source": s}) for s, t in library.items()])
retriever = InMemoryVectorStore.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": 2})
print(f"📚 {len(chunks)} chunks indexed")

## Step 2 — State, context and structured-output models

In [ ]:
import operator
from dataclasses import dataclass
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field

class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    notes: Annotated[list[dict], operator.add]     # filled in parallel by workers
    report: str
    feedback: str
    verdict: str
    revisions: int

@dataclass
class Context:
    reader_level: str = "beginner"                 # e.g. beginner / manager / expert

class Plan(BaseModel):
    sub_questions: list[str] = Field(description="2 to 4 focused sub-questions that together answer the main question")

class Review(BaseModel):
    verdict: Literal["pass", "revise"] = Field(description="pass if clear, grounded in the notes and well structured")
    feedback: str = Field(description="specific fixes if revise, else empty")

planner = llm.with_structured_output(Plan)
reviewer = llm.with_structured_output(Review)
MAX_REVISIONS = 2

## Step 3 — The nodes

Each node is small and does one job.

In [ ]:
from langgraph.types import interrupt, Send, Command
from langgraph.config import get_stream_writer
from langgraph.runtime import Runtime

def plan(state: ResearchState) -> dict:
    get_stream_writer()({"status": "🗂 planning"})
    p = planner.invoke(f"Break this research question into sub-questions: {state['question']}")
    return {"sub_questions": p.sub_questions[:4]}

def human_review(state: ResearchState) -> Command[Literal["research"]]:
    # ⏸ The human approves the plan, or sends back an edited list
    answer = interrupt({"message": "Approve the plan? Reply 'approve' or send an edited list.",
                        "sub_questions": state["sub_questions"]})
    subs = answer if isinstance(answer, list) else state["sub_questions"]
    # 🔀 one parallel research worker per sub-question
    return Command(update={"sub_questions": subs},
                   goto=[Send("research", {"sub_question": q}) for q in subs])

def research(inp: dict) -> dict:
    q = inp["sub_question"]
    get_stream_writer()({"status": f"🔎 researching: {q}"})
    docs = retriever.invoke(q)
    context = "\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
    note = llm.invoke(f"Using ONLY this context, answer in 2 sentences and keep the [source] tags.\n"
                      f"Context:\n{context}\n\nQuestion: {q}").content
    return {"notes": [{"sub_question": q, "note": note}]}

def write(state: ResearchState, runtime: Runtime[Context]) -> dict:
    get_stream_writer()({"status": "✍️ writing report" if not state.get("feedback") else "✍️ revising report"})
    notes = "\n".join(f"- Q: {n['sub_question']}\n  A: {n['note']}" for n in state["notes"])
    prompt = (f"Write a short research report (max 180 words) answering: {state['question']}\n"
              f"Audience: {runtime.context.reader_level}. Use markdown headings and bullet points. "
              f"Only use these notes and keep their [source] tags:\n{notes}")
    if state.get("feedback"):
        prompt += f"\n\nPrevious draft:\n{state['report']}\nFix this feedback: {state['feedback']}"
    return {"report": llm.invoke(prompt).content}

def review(state: ResearchState) -> dict:
    get_stream_writer()({"status": "🧪 reviewing"})
    r = reviewer.invoke(f"Question: {state['question']}\nReport:\n{state['report']}\n"
                        "Is it clear, structured, and does it cite sources?")
    return {"verdict": r.verdict, "feedback": r.feedback, "revisions": state.get("revisions", 0) + 1}

def after_review(state: ResearchState) -> Literal["write", "__end__"]:
    if state["verdict"] == "pass" or state["revisions"] >= MAX_REVISIONS:
        return END
    return "write"

## Step 4 — Assemble the graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy

retry = RetryPolicy(max_attempts=3)

builder = StateGraph(ResearchState, context_schema=Context)
builder.add_node("plan", plan, retry_policy=retry)
builder.add_node("human_review", human_review)
builder.add_node("research", research, retry_policy=retry)
builder.add_node("write", write, retry_policy=retry)
builder.add_node("review", review)

builder.add_edge(START, "plan")
builder.add_edge("plan", "human_review")
builder.add_edge("research", "write")          # waits for ALL research workers
builder.add_edge("write", "review")
builder.add_conditional_edges("review", after_review)

assistant = builder.compile(checkpointer=InMemorySaver())
show_graph(assistant)

## Step 5 — Run it: plan → pause for approval

We stream `"custom"` progress events and `"updates"` together.

In [ ]:
config = {"configurable": {"thread_id": "research-1"}}
ctx = Context(reader_level="beginner")

def run(inp):
    for mode, data in assistant.stream(inp, config, context=ctx, stream_mode=["custom", "updates"]):
        if mode == "custom":
            print(data["status"])
        elif "__interrupt__" in data:
            info = data["__interrupt__"][0].value
            print("\n⏸", info["message"])
            for i, q in enumerate(info["sub_questions"], 1):
                print(f"   {i}. {q}")

run({"question": "How do I build a reliable AI agent?", "revisions": 0})

## Step 6 — Approve (or edit) the plan and finish

Option 1: `Command(resume="approve")`
Option 2: send your own list, e.g. `Command(resume=["What makes agents fail?", "How do memory and tools help?"])`

In [ ]:
run(Command(resume="approve"))

final = assistant.get_state(config).values
print(f"\n✅ done — revisions: {final['revisions']}, verdict: {final['verdict']}\n")
print(final["report"])

## Step 7 — Inspect what happened

In [ ]:
print("Sub-questions researched in parallel:")
for n in final["notes"]:
    print(f"\n❓ {n['sub_question']}\n📝 {n['note']}")

print("\nCheckpoints saved:", len(list(assistant.get_state_history(config))))

## Step 8 — Try a different reader

Same graph, new thread, different **runtime context**.

In [ ]:
config = {"configurable": {"thread_id": "research-2"}}
ctx = Context(reader_level="busy manager — focus on risks and costs")
run({"question": "What should we check before putting an AI agent into production?", "revisions": 0})
run(Command(resume="approve"))
print("\n" + assistant.get_state(config).values["report"])

## 🚀 Make it yours — extension ideas

1. **Real web search:** replace the retriever with a search tool (e.g. Tavily) in `research`.
2. **Long-term memory:** save each user's preferred reader level in a `Store` (Day 14).
3. **Chat UI:** wrap `assistant` in FastAPI or Streamlit and stream `"messages"` tokens (Day 9, 15).
4. **Load your PDFs:** use `PyPDFLoader` to build the library from your own documents.
5. **Deploy:** move the code into `research_assistant.py` and run `langgraph dev` (Day 15).

## 🎓 Course complete!

| Days | You learned |
|---|---|
| 1–3 | State, nodes, edges, reducers, branches, loops, `Command` |
| 4–6 | OpenAI, messages, chatbots, tools, agents |
| 7–9 | Memory, human-in-the-loop, time travel, streaming, async |
| 10–12 | Structured output, workflow patterns, `Send`, subgraphs, multi-agent |
| 13–15 | Agentic RAG, long-term memory, production skills |
| 16 | A complete research assistant |

**Next steps:** build one small agent for a real problem you have, add tracing, and share it. 🚀